# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [38]:
%run ./setup_auth.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-02-11T11:23:57.816363",
    "last_interaction": "2026-02-11T11:23:57.816419",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": "",
    "saved_at": "2026-02-11T11:24:52.837929",
    "last_interaction": "2026-02-11T11:24:52.837985",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: b2yNqK0z5_IjrYp09UyL7OOjI1vxKgtgAEHnlf0qns0

Consumer DID: did:jwk:consumer

Consumer token: b2yNqK0z5_IjrYp09UyL7OOjI1vxKgtgAEHnlf0qns0


In [39]:
%run ./setup_catalog_policies.ipynb

{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:074161f3-bfe6-4bbe-a4ad-cec27de5c1ee",
    "dctIssued": "2026-02-11T11:25:15.903619Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalog": true,
    "dspaceParticipantId": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiIwLW5uempxdDNQVXZtMUVPaEVxNUNxQ05WcEVUM3I4S0Y3VWRKSDJlRFV1dDNhSFlFWGFxbG51SXZ4TUVmN0J1dkhMVnlUaW95U1BoMV9TaDF0ZnBwdF9aNXpXU1ZZUGp0ekZ0QlJsQ0pQYzNfc2YtelRIQURMMmNyYW84WmxXa1FFR2ZtX1g3Y2QtMkd5bmVuR0k0a01rLU9vRERUWEJNRmVoOVdtR1dyMGZBeHJ4Qkp3Unh4QTExODNIOWo1QkY3dlpVZ0Y3akYzV28xcVlva0Z4emJ0Q2FhWWkzTmE3aFYtWWRIYkg3QUhkYnlJa1JvTHhhVG9tb3ktRHRmb0dSSGRzTmpzNDFqZUwxTG5XczJENnVJMm43QTBFQ1hIaHJjZTlMbkt6eXNMWktRM3Ffa2hxR0UzRTZJT1djajRMT1NvNGJ4SXktZzNVT0NpeElSelJ2bFEifQ",
    "foafHomePage": null,
    "id": "urn:catalog:074161f3-bfe6-4bbe-a4ad-cec27de5c1ee"
}
{
    "catalogId": "urn:catalog:074161f3-bfe6-4bbe-a4ad-cec27de5c1ee",
    "dcatEndpointDescription": null,
    "dcatEndpo

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [21]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:5f4a246e-ee7c-4558-9004-75fe98183222",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:55190568-dfd5-47dd-8e4c-ef32c3fb9e10"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a4b0

## Provider creates initial offer (Provider -> Consumer)

In [22]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:5f4a246e-ee7c-4558-9004-75fe98183222",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:55190568-dfd5-47dd-8e4c-ef32c3fb9e10"
    },
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a94cf37a-49af-436f-be

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [23]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:5f4a246e-ee7c-4558-9004-75fe98183222",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:55190568-dfd5-47dd-8e4c-ef32c3fb9e10"
    },
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a4b0e6b0-908a-430b-b8b2-9d4f9f192ea2",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [24]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:5f4a246e-ee7c-4558-9004-75fe98183222",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:55190568-dfd5-47dd-8e4c-ef32c3fb9e10"
    },
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a94cf37a-49af-436f-be79-39040f3a6aca",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [25]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a4b0e6b0-908a-430b-b8b2-9d4f9f192ea2",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-16T07:28:47.724118Z",
    "updatedAt": "2026-02-16T07:28:48.498077Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:4

## Provider creates the Agreement (Provider -> Consumer)

In [26]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a94cf37a-49af-436f-be79-39040f3a6aca",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-16T07:28:47.590947Z",
    "updatedAt": "2026-02-16T07:28:48.727881Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:f642b

## Consumer verifies the agreement (Consumer -> Provider)

In [27]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a4b0e6b0-908a-430b-b8b2-9d4f9f192ea2",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-16T07:28:47.724118Z",
    "updatedAt": "2026-02-16T07:28:49.640982Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:4

## Provider finalizes the negotiation (Provider -> Consumer)

In [28]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f642b1db-b49c-4daa-aaf7-c8b8b7717818",
    "providerPid": "urn:provider-pid:46b85c6b-3e26-45e1-890b-01fdf857beda",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:a94cf37a-49af-436f-be79-39040f3a6aca",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-16T07:28:47.590947Z",
    "updatedAt": "2026-02-16T07:28:50.114042Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid

## Final agreement

In [29]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
  "negotiationAgentProcessId": "urn:negotiation-process:a94cf37a-49af-436f-be79-39040f3a6aca",
  "negotiationAgentMessageId": "urn:negotiation-message:16b7c32e-6556-496e-85d3-2ed28968231f",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:55190568-dfd5-47dd-8e4c-ef32c3fb9e10",
    "timestamp": "1771226928"
  },
  "target": "urn:dataset:55190568-dfd5-47dd-8e4c-ef32c3fb9e10",
  "state": "ACTIVE",
  "createdAt": "2026-02-16T07:28:48.739325Z",
  "updatedAt": "2026-02-16T07:28:50.137129Z"
}

Final agreement id: 
urn:agreement:80df1059-0eef-439c-87c2-a7f122993070



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [30]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
    "format": "http+asd",
    "dataAddress": null,
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:b681784f-0a85-4589-b55f-54612659ee64",
    "state": "REQUESTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
    "callbackAddress": "http://127.0.0.1:

## Start transfer (Provider -> Consumer)

In [31]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "dataAddress": {
        "@type": "DataAddress",
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://example.com",
        "endpointProperties": [
            {
                "@type": "EndpointProperty",
                "name": "authorization",
                "value": "TOKEN-ABCDEFG"
            },
            {
                "@type": "EndpointProperty",
                "name": "authType",
                "value": "bearer"
            }
        ]
    },
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "dataAddress": {
      "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
      "endpoint": "http://example.com",
      "endpointProperties": [
        {
          "name": "authorization",
          "value": "TOKEN-ABCDEFG"
        },
        {
          "name": "authType",
          "value": "bearer"
        }
      ]
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "state": "STARTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:fd7e9192-ebd2-4ece-bd76-20421af1bc4c",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:

## Suspend transfer (Consumer -> Provider)

In [32]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:b681784f-0a85-4589-b55f-54612659ee64",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
   

## Restart transfer (Consumer -> Provider)

In [33]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "dataAddress": null
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "state": "STARTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:b681784f-0a85-4589-b55f-54612659ee64",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [34]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:fd7e9192-ebd2-4ece-bd76-20421af1bc4c",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
   

## Failure Test: Attempt start with invalid parameters

In [35]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "dataAddress": None
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "dataAddress": null
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "HTTP Error 400 Bad Request: {\"@context\":[\"https://w3id.org/dspace/2025/1/context.jsonld\"],\"@type\":\"TransferError\",\"consumerPid\":null,\"providerPid\":null,\"code\":\"6030\",\"reason\":[\"TransferProcessMessageType TransferStartMessage is not allowed here. Current state is SUSPENDED ByProvider\",\"Failed to parse file\"]}"
    ]
  }
}


## Failure Test: Attempt duplicate or invalid suspension

In [36]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "6030",
    "reason": [
      "TransferProcessMessageType TransferSuspensionMessage is not allowed here. Current state is SUSPENDED",
      "Failed to parse file"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [37]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:1040192c-3a34-42cd-9ec9-6879349ea6d0",
    "providerPid": "urn:provider-pid:2ffa6ad2-c966-408a-af70-8c452f6449c2",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:fd7e9192-ebd2-4ece-bd76-20421af1bc4c",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+asd",
    "agreementId": "urn:agreement:80df1059-0eef-439c-87c2-a7f122993070",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-16T07:28:51.330101Z",